<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z340_NormalizacionSeries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Normalización de series para agrupar por forma

El problema: queremos que un modelo aprenda patrones de **forma** (sube, baja, estable) sin que la escala lo confunda. Un producto de 1 tn y otro de 1000 tn con la misma forma deberían generar el mismo aprendizaje.

El tema con los ceros: la estandarización z-score divide por la std. Si una serie tiene muchos ceros la std es grande pero el nivel es bajo, y la normalización pierde sentido.

## Las 3 normalizaciones

### 1. Dividir por máximo
```
serie_norm = serie / max(serie)
```
- Escala todo a [0, 1]. El pico histórico siempre vale 1.0.
- Robusto a ceros: el máximo siempre es > 0 si hubo alguna venta.
- Todos los productos quedan comparables en forma relativa a su propio pico.
- **Desnormalizar**: `pred_real = pred_norm × max(serie_original)`

### 2. Módulo (norma L2)
```
serie_norm = serie / ||serie||   donde ||serie|| = sqrt(Σ tn²)
```
- Convierte la serie en un **vector unitario** (norma = 1).
- Captura la distribución relativa de ventas entre períodos: si vendiste más en algunos meses que en otros, eso queda representado sin importar el nivel absoluto.
- Robusto a ceros: contribuyen 0 a la norma.
- **Desnormalizar**: `pred_real = pred_norm × ||serie_original||`

### 3. Serie indexada
```
serie_norm = serie / base   donde base = media primeros 3 meses con venta
```
- Todas las series 'arrancan' en 1.0 (el nivel inicial es la referencia).
- Un valor de 0.5 significa 'la mitad del nivel inicial'; 2.0 significa 'el doble'.
- Captura el **cambio relativo** respecto al origen histórico.
- Robusto a ceros en el presente: el base se calcula sobre los primeros meses.
- **Desnormalizar**: `pred_real = pred_norm × base`

## Pipeline
1. Normalizar cada serie
2. Construir features de lags sobre la serie normalizada (todos los productos juntos)
3. Entrenar LightGBM sobre esa feature matrix
4. Predecir el valor normalizado para 202002
5. Desnormalizar → pred_real
6. Submit

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle lightgbm

In [ ]:
import os, shutil
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'competencia':    'labo-iii-2026-rosario',
    'periodo_corte':  201910,
    'periodo_target': 201912,
    'horizonte':      2,
    # lags para features
    'lags':           [1, 2, 3, 6, 12],
    # LightGBM
    'lgb_params': {
        'objective':       'regression',
        'metric':          'rmse',
        'n_estimators':    300,
        'learning_rate':   0.05,
        'num_leaves':      31,
        'min_child_samples': 10,
        'verbose':         -1,
    },
    'drive_path': '/content/buckets/b1/exp/NormalizacionSeries',
}

os.makedirs(PARAM['drive_path'], exist_ok=True)

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

# Funciones de normalización / desnormalización

In [ ]:
def norm_max(serie):
    """Divide por el máximo histórico. Escala [0,1]. Robusto a ceros."""
    m = serie.max()
    if m <= 0:
        return serie.copy(), 1.0
    return serie / m, float(m)


def norm_l2(serie):
    """Divide por la norma L2 (módulo del vector). Serie unitaria."""
    norma = float(np.sqrt((serie ** 2).sum()))
    if norma <= 0:
        return serie.copy(), 1.0
    return serie / norma, norma


def norm_index(serie, n_base=3):
    """
    Indexada: divide por la media de los primeros `n_base` meses con venta > 0.
    El nivel inicial = 1.0. Captura cambio relativo al origen.
    """
    positivos = serie[serie > 0]
    if len(positivos) == 0:
        return serie.copy(), 1.0
    base = float(positivos[:n_base].mean())
    if base <= 0:
        return serie.copy(), 1.0
    return serie / base, base


NORMALIZACIONES = {
    'max':   norm_max,
    'l2':    norm_l2,
    'index': norm_index,
}

print('Funciones OK')

# Visualización — efecto de cada normalización en 6 productos

In [ ]:
np.random.seed(42)
muestra = np.random.choice(productos, 6, replace=False).tolist()

fig, axes = plt.subplots(4, 6, figsize=(18, 10))
titulos_filas = ['Original', 'Norm max', 'Norm L2', 'Norm index']

for col, pid in enumerate(muestra):
    serie = (
        tb_ventas.filter(pl.col('product_id') == pid)
        .sort('periodo')['tn'].to_numpy().astype(float)
    )

    normas = [
        serie,
        norm_max(serie)[0],
        norm_l2(serie)[0],
        norm_index(serie)[0],
    ]
    colores = ['steelblue', 'tomato', 'green', 'purple']

    for row, (s_norm, color) in enumerate(zip(normas, colores)):
        ax = axes[row][col]
        ax.plot(s_norm, color=color, linewidth=1.2)
        ax.fill_between(range(len(s_norm)), s_norm, alpha=0.15, color=color)
        if col == 0:
            ax.set_ylabel(titulos_filas[row], fontsize=8)
        if row == 0:
            ax.set_title(f'pid {pid}', fontsize=7)
        ax.tick_params(labelsize=6)

fig.suptitle('Efecto de cada normalización — mismas 6 series', fontsize=10)
plt.tight_layout()
plt.show()

# Pipeline — construir features + entrenar LightGBM + predecir

In [ ]:
def build_features(tb, productos, normalizacion_fn, lags, periodo_corte=None):
    """
    Construye la feature matrix para todos los productos.
    Si periodo_corte != None, usa solo hasta ese período.
    Devuelve X, y, escalas (para desnormalizar), y los product_ids por fila.
    """
    rows = []
    for pid in productos:
        df = tb.filter(pl.col('product_id') == pid).sort('periodo')
        if periodo_corte is not None:
            df = df.filter(pl.col('periodo') <= periodo_corte)
        serie = df['tn'].to_numpy().astype(float)

        if len(serie) < max(lags) + 1:
            continue

        serie_norm, escala = normalizacion_fn(serie)
        max_lag = max(lags)

        for t in range(max_lag, len(serie_norm)):
            feat = {f'lag_{l}': serie_norm[t - l] for l in lags}
            feat['target']     = serie_norm[t]
            feat['product_id'] = pid
            feat['escala']     = escala
            feat['t']          = t
            rows.append(feat)

    return pl.DataFrame(rows)


def predecir_productos(tb_full, productos, normalizacion_fn, lags,
                        periodo_corte, modelo_lgb, horizonte=2):
    """
    Para cada producto: normaliza, predice recursivamente horizonte pasos,
    desnormaliza y devuelve la predicción final.
    """
    max_lag = max(lags)
    preds   = []

    for pid in productos:
        serie = (
            tb_full.filter(
                (pl.col('product_id') == pid) &
                (pl.col('periodo') <= periodo_corte)
            ).sort('periodo')['tn'].to_numpy().astype(float)
        )

        if len(serie) < max_lag:
            preds.append({'product_id': pid, 'tn': max(float(serie.mean()), 0.0)})
            continue

        serie_norm, escala = normalizacion_fn(serie)
        s = list(serie_norm)

        # predicción recursiva: horizonte pasos
        for _ in range(horizonte):
            feat = np.array([[s[-(l)] for l in lags]])
            p_norm = float(modelo_lgb.predict(feat)[0])
            p_norm = max(p_norm, 0.0)
            s.append(p_norm)

        pred_real = max(s[-1] * escala, 0.0)
        preds.append({'product_id': pid, 'tn': pred_real})

    return pl.DataFrame(preds)


print('Pipeline OK')

# Backtesting — las 3 normalizaciones

In [ ]:
tb_real = (
    tb_ventas.filter(pl.col('periodo') == PARAM['periodo_target'])
    .select(['product_id','tn']).rename({'tn':'tn_real'})
)

rmse_bt = {}
modelos_entrenados = {}

for nombre, fn in NORMALIZACIONES.items():
    print(f"\n── {nombre} ──────────────")

    # features de entrenamiento (hasta periodo_corte)
    tb_feat = build_features(
        tb_ventas, productos, fn,
        lags=PARAM['lags'],
        periodo_corte=PARAM['periodo_corte']
    )
    feat_cols = [f'lag_{l}' for l in PARAM['lags']]
    X = tb_feat[feat_cols].to_numpy()
    y = tb_feat['target'].to_numpy()

    print(f"  filas de entrenamiento: {len(X):,}")

    # entrenar LightGBM
    modelo = lgb.LGBMRegressor(**PARAM['lgb_params'])
    modelo.fit(X, y)
    modelos_entrenados[nombre] = modelo

    # predecir 201912
    tb_pred = predecir_productos(
        tb_ventas, productos, fn,
        lags=PARAM['lags'],
        periodo_corte=PARAM['periodo_corte'],
        modelo_lgb=modelo,
        horizonte=PARAM['horizonte']
    )

    tb_bt = tb_real.join(tb_pred, on='product_id', how='left')
    err   = (tb_bt['tn_real'] - tb_bt['tn']).to_numpy()
    rmse  = float(np.sqrt((err ** 2).mean()))
    rmse_bt[nombre] = rmse
    print(f"  RMSE backtesting 201912: {rmse:.4f}")

print("\n── Resumen ──")
for n, r in sorted(rmse_bt.items(), key=lambda x: x[1]):
    print(f"  {n:8s}: {r:.4f}")

# Visualización — predicciones vs real en backtesting

In [ ]:
# recalcular preds para graficar
preds_bt = {}
for nombre, fn in NORMALIZACIONES.items():
    tb_pred = predecir_productos(
        tb_ventas, productos, fn,
        lags=PARAM['lags'],
        periodo_corte=PARAM['periodo_corte'],
        modelo_lgb=modelos_entrenados[nombre],
        horizonte=PARAM['horizonte']
    ).rename({'tn': f'pred_{nombre}'})
    preds_bt[nombre] = tb_pred

tb_comp = tb_real
for nombre, df in preds_bt.items():
    tb_comp = tb_comp.join(df, on='product_id', how='left')

# scatter real vs predicho para cada normalización
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colores = {'max': 'tomato', 'l2': 'green', 'index': 'purple'}

for i, nombre in enumerate(['max', 'l2', 'index']):
    real  = tb_comp['tn_real'].to_numpy()
    pred  = tb_comp[f'pred_{nombre}'].to_numpy()
    rmse  = rmse_bt[nombre]

    ax = axes[i]
    # cap para visualización
    cap = np.percentile(np.concatenate([real, pred]), 95)
    mask = (real <= cap) & (pred <= cap)
    ax.scatter(real[mask], pred[mask], s=8, alpha=0.4, color=colores[nombre])
    ax.plot([0, cap], [0, cap], 'k--', linewidth=0.8)
    ax.set_xlabel('tn real')
    ax.set_ylabel('tn predicho')
    ax.set_title(f'Norm {nombre}\nRMSE={rmse:.4f}', fontsize=9)

fig.suptitle('Backtesting 201912 — real vs predicho por normalización', fontsize=10)
plt.tight_layout()
plt.show()

# Submit — las 3 normalizaciones con datos completos hasta 201912

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

# reentrenar con toda la historia hasta 201912 antes de predecir 202002
PERIODO_FINAL = 201912

for nombre, fn in NORMALIZACIONES.items():
    print(f"\n── submit {nombre} ──")

    # reentrenar con todos los datos hasta 201912
    tb_feat = build_features(
        tb_ventas, productos, fn,
        lags=PARAM['lags'],
        periodo_corte=PERIODO_FINAL
    )
    feat_cols = [f'lag_{l}' for l in PARAM['lags']]
    X = tb_feat[feat_cols].to_numpy()
    y = tb_feat['target'].to_numpy()

    modelo = lgb.LGBMRegressor(**PARAM['lgb_params'])
    modelo.fit(X, y)

    # predecir 202002 (horizonte=2 desde 201912)
    tb_final = predecir_productos(
        tb_ventas, productos, fn,
        lags=PARAM['lags'],
        periodo_corte=PERIODO_FINAL,
        modelo_lgb=modelo,
        horizonte=PARAM['horizonte']
    )

    archivo = f'lgbm_norm_{nombre}.csv'
    mensaje = f'LightGBM norm={nombre} lags={PARAM["lags"]} RMSE_bt={rmse_bt[nombre]:.4f}'

    tb_final.write_csv(archivo)
    shutil.copy(archivo, f"{PARAM['drive_path']}/{archivo}")
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'  submitted: {archivo}')
    print(f'  guardado en Drive: {PARAM["drive_path"]}/{archivo}')